# Document-oriented Practice Lab
## Anomaly Detection Methods Database

**Master IASD — DataAES — 2025**  
Paul Boniol

---

This notebook guides you through a MongoDB lab on a dataset of **133 anomaly detection methods**
and **13 algorithmic families** from [TSADtaxonomy](https://boniolp.github.io/TSADtaxonomy/).

| | |
|---|---|
| **Student name** | _____________________________________________ |
| **Student seed (Part 2)** | __________ |

---

> ⚠️ **The database contains inconsistencies.** Some words have inconsistent capitalization (Sup or sup), and some names are either fully written out or use only the initial (Themis Palpanas or T. Palpanas). Despite these small inconsistencies, they all refer to the same entity (e.g., Themis Palpanas and T. Palpanas are the same person, and Sup and sup are the same category).


## 0 · Setup

Run the cell below once to install pymongo (skip if already installed).


In [2]:
from pymongo import MongoClient, ASCENDING, DESCENDING
import json, glob, os, hashlib, random, pprint

# Connect to a local MongoDB instance (default port 27017)
client = MongoClient("mongodb://localhost:27017/")
db = client["tsad_lab"]

print("Connected — MongoDB version:", client.server_info()["version"])


Connected — MongoDB version: 8.0.34


## Hash utility functions

For each response that output a list of documents, please print the correspodning hash.

In [3]:
def _canonical(obj):
    """Recursively strip _id and convert non-JSON types to str."""
    if isinstance(obj, dict):
        return {k: _canonical(v) for k, v in sorted(obj.items()) if k != '_id'}
    if isinstance(obj, list):
        return [_canonical(i) for i in obj]
    if isinstance(obj, (bool, int, float, str, type(None))):
        return obj
    return str(obj)   # ObjectId, datetime, Decimal128 …

def result_hash(result):
    """16-hex fingerprint, list sorted (use for unordered results)."""
    c = _canonical(result)
    if isinstance(c, list):
        c = sorted(c, key=lambda x: json.dumps(x, sort_keys=True, ensure_ascii=False))
    return hashlib.sha256(
        json.dumps(c, sort_keys=True, ensure_ascii=False).encode()
    ).hexdigest()[:16]

def ordered_hash(result):
    """16-hex fingerprint, preserves list order (use for sorted/ranked results)."""
    c = _canonical(result)
    return hashlib.sha256(
        json.dumps(c, sort_keys=True, ensure_ascii=False).encode()
    ).hexdigest()[:16]

print("Hash utilities ready.")


Hash utilities ready.


## 0 · Data Loading (Part 1)

Run this cell once to load the base dataset.  
Set `DATA_DIR` to the folder that contains the JSON files.


In [4]:
DATA_DIR = "./data"   # ← adjust if needed

methods, families = [], []
for fp in sorted(glob.glob(os.path.join(DATA_DIR, '*.json'))):
    doc = json.load(open(fp, encoding='utf-8'))
    (methods if 'category' in doc else families).append(doc)

db.methods.drop()
db.families.drop()
db.methods.insert_many(methods)
db.families.insert_many(families)

print(f"Loaded {db.methods.count_documents({})} methods "
      f"and {db.families.count_documents({})} families.")


Loaded 133 methods and 13 families.


---
# Part 1 — Getting Started *(in-class)*

These questions are completed **during the lab session**.  
Write your pymongo query in the code cell below each question.


## Section A — Inserting and Exploring the Data


**Q1.** Verify the data insertion by listing **all** documents from each collection.

In [5]:
# Your answer here. Print result_hash for each collection.
result_methods = list(db.methods.find())
result_families = list(db.families.find())
print(result_hash(result_families), result_hash(result_methods))


97ed2a51ddeeb17d bca6c9f319ca63a9


**Q2.** Count the total number of documents in the `methods` collection.

In [6]:
# Your answer here
print(db.methods.count_documents( {}))

133


**Q3.** Count the total number of documents in the `families` collection.

In [7]:
# Your answer here
print(db.families.count_documents({}))


13


**Q4.** List all **distinct category names** present in the `methods` collection.

In [8]:
# Your answer here
categories = db.methods.distinct("category")
pprint.pprint(categories)
print(result_hash(categories))

['Clustering-based',
 'Discord-Based',
 'Discord-based',
 'Discord-based and Matrix-Profile-based',
 'Distance-based',
 'Distribution-based',
 'Encoding-based',
 'Forecasting-based',
 'Graph-Based',
 'Graph-based',
 'Proximity-based',
 'Reconstruction-based',
 'Tree-based']
65b49b8cf056a891


**T1.** Suppose that, instead of MongoDB, you must store the same data
(the methods and families collections) in a
relational database (e.g. PostgreSQL).


**(a)**. Propose a relational schema able to store these data. For each table, give its attributes and their types, its **primary key**,
its **foreign keys**, and any relevant integrity constraints

**(b)** Justify your design. Which fields of the JSON documents
cannot be stored *as is* in a relational table, and how do you handle them?
Which normal form does your schema satisfy? How does your schema handle the
inconsistencies mentioned in Section~2?

*Your answer here*

## table methodes :

- **primary key :**
  - `methode_id` : int and not null

- **foreign_key:**
  - `name` : str
  - `full name` : str
  - `category` : int in family_id of table families
  - `Dim` : str in "Univariate" or "Multivariate"
  - `Sup` : str in "Unsupervised", "Semi-supervised" or "Supervised"
  - `Stream` : bool
  - `year` : int
  - `paper` : str
  - `description` : str
  - `url` : str
  - `code` : str
  - `snippet` : str

### table Author_methods :

- **primary key :**
  - `author_name` : str not null
  - `method_id` : int not null

### table families :

- **primary key :**
  - `family_id` : int not null
- **foreign key :**
  - `name` : str
  - `description` : str
  - `year` : int

### table author_alternative_name :

- **primary_key :**
  - `name` : str

- **foreign_key :**
  - `real_name`

## B/

first we make it so the 3 normal form are respected,

- **1st normal form :** no list (so we create an association table for author and method) and choose our primary key
- **2nd normal** is true because only 1 primary key each time
- and **3rd** is verified here because everything directly depend on primary key

Finally we deal with inconsistency by puting method_id and family_id as integer, by restricting the values of Dim and sup and finally for the inconsistency in author name, we therefore create a new_table to associate real_name with all the variants.

## Section B — Filtering and Projection


**Q5.** Find all methods that handle **Multivariate** time series. Display only `name` and `category`.

In [9]:
# Your answer here. Print result_hash of the result.
result_5 = list(db.methods.find({"Dim": "Multivariate"},{"name": True, "category" : True, "_id": False}))

pprint.pprint(result_5)
print(result_hash(result_5))

[{'category': 'Forecasting-based', 'name': 'AD-LTI'},
 {'category': 'Reconstruction-based', 'name': 'AE'},
 {'category': 'Distribution-based', 'name': 'AOSVM'},
 {'category': 'Reconstruction-based', 'name': 'Anomaly-Transformer'},
 {'category': 'Forecasting-based', 'name': 'Autoformer'},
 {'category': 'Clustering-based', 'name': 'CBLOF'},
 {'category': 'Proximity-based', 'name': 'COF'},
 {'category': 'Distribution-based', 'name': 'COPOD'},
 {'category': 'Forecasting-based', 'name': 'CoalESN'},
 {'category': 'Distribution-based', 'name': 'ConInd'},
 {'category': 'Reconstruction-based', 'name': 'D3R'},
 {'category': 'Clustering-based', 'name': 'DBSCAN'},
 {'category': 'Proximity-based', 'name': 'DBStream'},
 {'category': 'Reconstruction-based', 'name': 'DCdetector'},
 {'category': 'Proximity-based', 'name': 'DILOF'},
 {'category': 'Forecasting-based', 'name': 'DeepAnT'},
 {'category': 'Reconstruction-based', 'name': 'DeepNAP'},
 {'category': 'Encoding-based', 'name': 'DeepPCA'},
 {'categ

**Q6.** Find all methods published **after 2018** that support **streaming** (`Stream = True`). Show `name`, `year`, and `category` only.

In [10]:
# Your answer here. Print result_hash of the result.
result_6 = list(db.methods.find({"Stream": True, "year" : {"$gt" : 2018}}, {"name": True, "year":True, "category": True, "_id" : False}))
pprint.pprint(result_6)
print(result_hash(result_6))

[{'category': 'Forecasting-based', 'name': 'AD-LTI', 'year': 2020},
 {'category': 'Forecasting-based', 'name': 'Autoformer', 'year': 2021},
 {'category': 'Forecasting-based', 'name': 'Chronos', 'year': 2024},
 {'category': 'Reconstruction-based', 'name': 'D3R', 'year': 2023},
 {'category': 'Discord-Based', 'name': 'DAMP', 'year': 2022},
 {'category': 'Reconstruction-based', 'name': 'DiffAD', 'year': 2023},
 {'category': 'Reconstruction-based', 'name': 'DiffusionAE', 'year': 2023},
 {'category': 'Reconstruction-based', 'name': 'ImDiffusion', 'year': 2023},
 {'category': 'Discord-based', 'name': 'LAMP', 'year': 2020},
 {'category': 'Reconstruction-based', 'name': 'MAD-GAN', 'year': 2019},
 {'category': 'Reconstruction-based', 'name': 'MLLM4TS', 'year': 2025},
 {'category': 'Forecasting-based', 'name': 'MTAD-GAT', 'year': 2020},
 {'category': 'Forecasting-based', 'name': 'Moirai', 'year': 2024},
 {'category': 'Reconstruction-based', 'name': 'OFA', 'year': 2023},
 {'category': 'Forecasting

**Q7.** Find the **5 most recently published** methods. Show `name`, `year`, and `category`. Sort them first by year descending then by name ascending.

In [11]:
# Your answer here. Print ordered_hash of the result.
result_7 = list(
    db.methods.find({},{"_id": False, "name": True, "year": True, "category": True}).sort([("year", -1), ("name", 1)]).limit(5)
                 )

pprint.pprint(result_7)
print(ordered_hash(result_7))

[{'category': 'Reconstruction-based', 'name': 'MLLM4TS', 'year': 2025},
 {'category': 'Forecasting-based', 'name': 'Chronos', 'year': 2024},
 {'category': 'Reconstruction-based', 'name': 'MOMENT', 'year': 2024},
 {'category': 'Forecasting-based', 'name': 'Moirai', 'year': 2024},
 {'category': 'Reconstruction-based', 'name': 'SARAD', 'year': 2024}]
d9fcc94c235dc931


**Q8.** Find all methods whose `paper` field contains the word **"neural"** (case-insensitive).

In [12]:
# Your answer here. Print result_hash of the result.
result_8 = list(db.methods.find({
    "paper" : {"$regex": r"\bneural\b", "$options" : "i"}
})) 

pprint.pprint(result_8)
print(result_hash(result_8))

[{'Dim': 'Multivariate',
  'Stream': True,
  'Sup': 'Unsupervised',
  '_id': ObjectId('6ae12fe246964372a9e7628d'),
  'authors': ['Chunggyeom Kim',
              'Jinhyuk Lee',
              'Raehyun Kim',
              'Youngbin Park',
              'Jaewoo Kang'],
  'bibtex': '@article{10.1016/j.ins.2018.05.020,author = {Kim, Chunggyeom and '
            'Lee, Jinhyuk and Kim, Raehyun and Park, Youngbin and Kang, '
            'Jaewoo},title = {DeepNAP: Deep neural anomaly pre-detection in a '
            'semiconductor fab},year = {2018},issue_date = {Aug '
            '2018},publisher = {Elsevier Science Inc.},address = {USA},volume '
            '= {457},number = {C},issn = {0020-0255},doi = '
            '{10.1016/j.ins.2018.05.020},journal = {Inf. Sci.},month = '
            'aug,pages = {1–11},numpages = {11},}',
  'category': 'Reconstruction-based',
  'code': '',
  'description': 'DeepNAP is an anomaly pre-detection model based on recurrent '
                 'neural networks. 

**Q9.** Find all methods that have a code repository (the `code` field exists *and* is not an empty string).

In [13]:
# Your answer here. Print result_hash of the result.
result_9 = list(db.methods.find({
    "code" : {"$exists" : True, "$ne": ""}
}))
pprint.pprint(result_9)
print(result_hash(result_9))

[{'Dim': 'Multivariate',
  'Stream': False,
  'Sup': 'Semi-Supervised',
  '_id': ObjectId('6ae12fe246964372a9e76273'),
  'authors': ['Mayu Sakurada', 'Takehisa Yairi'],
  'bibtex': '@inproceedings{10.1145/2689746.2689747,author = {Sakurada, Mayu '
            'and Yairi, Takehisa},title = {Anomaly Detection Using '
            'Autoencoders with Nonlinear Dimensionality Reduction},year = '
            '{2014},isbn = {9781450331593},publisher = {Association for '
            'Computing Machinery},address = {New York, NY, USA},doi = '
            '{10.1145/2689746.2689747},booktitle = {Proceedings of the MLSDA '
            '2014 2nd Workshop on Machine Learning for Sensory Data '
            'Analysis},pages = {4–11},numpages = {8},location = {Gold Coast, '
            "Australia QLD, Australia},series = {MLSDA'14}}",
  'category': 'Reconstruction-based',
  'code': 'https://github.com/TheDatumOrg/TSB-UAD',
  'description': 'Autoencoder is a type of artificial neural network used to '
  

**Q10.** Find all methods that do **not** have a `snippet` field. Display only their names.

In [14]:
# Your answer here. Print result_hash of the result.
result_10 = list(db.methods.find({"snippet" : {"$exists" : False}}, {"name": True, "_id": False}))
pprint.pprint(result_10)
print(result_hash(result_10))

[{'name': 'AD-LTI'},
 {'name': 'AMD'},
 {'name': 'AOSVM'},
 {'name': 'ARIMA'},
 {'name': 'Anomaly-Transformer'},
 {'name': 'Autoformer'},
 {'name': 'BAGEL'},
 {'name': 'CAE'},
 {'name': 'CBLOF'},
 {'name': 'COF'},
 {'name': 'COPOD'},
 {'name': 'Chronos'},
 {'name': 'CoalESN'},
 {'name': 'ConInd'},
 {'name': 'D3R'},
 {'name': 'DAD'},
 {'name': 'DADS'},
 {'name': 'DAMP'},
 {'name': 'DBSCAN'},
 {'name': 'DBStream'},
 {'name': 'DCdetector'},
 {'name': 'DES'},
 {'name': 'DILOF'},
 {'name': 'DONUT'},
 {'name': 'DeepLSTM'},
 {'name': 'DeepNAP'},
 {'name': 'DeepPCA'},
 {'name': 'DeepSVM'},
 {'name': 'DiffAD'},
 {'name': 'DiffusionAE'},
 {'name': 'EDBN'},
 {'name': 'EM-HMM'},
 {'name': 'ES'},
 {'name': 'ESN'},
 {'name': 'Enc-Dec-AD'},
 {'name': 'Ensemble-GI'},
 {'name': 'Eros-SVMs'},
 {'name': 'Extended-IForest'},
 {'name': 'FastMCD'},
 {'name': 'GAN'},
 {'name': 'GPT4TS'},
 {'name': 'GeckoFSM'},
 {'name': 'GrammarViz'},
 {'name': 'HMAD'},
 {'name': 'HOTSAX'},
 {'name': 'HSDE'},
 {'name': 'Heal

**Q11.** Find methods published between **2015 and 2020 inclusive** that are **Unsupervised**. Sort by year ascending then name ascending.

In [35]:
# Your answer here. Print ordered_hash of the result.
result_11 = list(db.methods.find(
    {"year": {"$gt" : 2014, "$lt" : 2021}, "$or" : [{"Sup" : {"$regex": "^unsupervised$", "$options": "i"}}, 
   {"sup": {"$regex": "^unsupervised$", "$options": "i"}}] }
).sort([("year", 1), ("name", 1)]))
pprint.pprint(result_11)
print(ordered_hash(result_11))


[{'Dim': 'Univariate',
  'Stream': False,
  'Sup': 'Unsupervised',
  '_id': ObjectId('6ae12fe246964372a9e7629e'),
  'authors': ['Pavel Senin',
              'Jessica Lin',
              'Xing Wang',
              'Tim Oates',
              'Sunil Gandhi'],
  'bibtex': '@inproceedings{Senin2015TimeSA,title={Time series anomaly '
            'discovery with grammar-based compression},author={Pavel Senin and '
            'Jessica Lin and Xing Wang and Tim Oates and S. Gandhi and Arnold '
            'P. Boedihardjo and Crystal Chen and Susan '
            'Frankenstein},booktitle={International Conference on Extending '
            'Database Technology},year={2015},}',
  'category': 'Encoding-based',
  'citations': 1825,
  'code': 'https://github.com/GrammarViz2/grammarviz2_src',
  'description': 'GrammarViz approach for time series pattern discovery is '
                 'based on two algorithms that have linear time and space '
                 'complexity: (i) Symbolic Aggregate appro

**Q12.** Find all methods where **Eamonn Keogh** is listed as an author.

In [16]:
# Your answer here. Print ordered_hash of the result.
result_12= list(db.methods.find({
    "authors": {
        "$regex": r"^\s*Eamonn\s+Keogh\s*$",
        "$options": "i"
    }
}))

pprint.pprint(result_12)
print(ordered_hash(result_12))

[{'Dim': 'Univariate',
  'Stream': False,
  'Sup': 'Unsupervised',
  '_id': ObjectId('6ae12fe246964372a9e76282'),
  'authors': ['Dragomir Yankov', 'Eamonn Keogh', 'Umaa Rebbapragada'],
  'bibtex': '@article{Yankov2008,\n'
            '  author = {Yankov, Dragomir and Keogh, Eamonn and Rebbapragada, '
            'Umaa},\n'
            '  title = {Disk aware discord discovery: Finding unusual time '
            'series in terabyte sized datasets},\n'
            '  journal = {Knowledge and Information Systems},\n'
            '  volume = {17},\n'
            '  pages = {241--262},\n'
            '  year = {2008},\n'
            '  month = {11},\n'
            '  doi = {10.1007/s10115-008-0131-9}\n'
            '}',
  'category': 'Discord-based',
  'code': '',
  'description': 'Disk Aware Discord Discovery is an exact algorithm for '
                 'finding anomalous subsequences (‘discords’) in time series '
                 'that are too large to fit in memory. It detects discords '


**T2**. We now imagine a much larger version of the `methods` collection
(e.g. hundreds of millions of documents describing papers), which no longer fits
on a single server and must therefore be **distributed (sharded)** over
several servers. We assume that the **most frequent queries** retrieve papers
published within a given **time range**.

**(a)** What is the most commonly used sharding strategy in MongoDB?
Describe how documents are distributed among the servers with this strategy, and
how the query above would be executed.

**(b)** Which sharding strategy would you
choose for this workload? Justify your choice by comparing it with the strategy of (a).

**(c)** What are the limits or risks of the strategy you proposed in (b),
and how could you mitigate them?

a/ I didn't find what was the most used strategy but let's say it is s hash sarding : according to an hash functions, documents are putted in a chunk and each chunk is put in a shard. Then when a query arrive, if the hash function is calculable you can calculate it and know in whch shard it is going to be


B/ i would choose for the documents a sharded according to some range, for example year of publication. for example year 2018-1019 will be in one chunk, 2020-2021 in another etc ... in this case, the router only transmit the query to shard that are concern. therefore for query about year > 2018, it would only concern shard that contains data > 2018 

c/ some years might contains way more data than others, also new years might be more requested too. to deal with that, we can use a range on _id too by using range on _id that are not too big (other way, all new data which are all the new article would still be on the same chunk and shard)



*Your answer here*

## Section C — Aggregation


**Q13.** Count the number of methods **per category**. Sort from the largest to the smallest group. In case of equality, sort by lexicographical order based on the category name.

In [ ]:
pipeline = [
    {"$group": {"_id": {"$toLower": {"$trim": {"input": "$category"}}}, "count": {"$sum": 1}}},
    {"$sort": {"count": -1, "_id": 1}}
]

result_13 = list(db.methods.aggregate(pipeline))

pprint.pprint(result_13)
print(ordered_hash(result_13))


**Q14.** Find all categories that contain **more than 10 methods**. Sort from the largest to the smallest group.

In [ ]:
pipeline = [
    {"$group": {"_id": {"$toLower": {"$trim": {"input": "$category"}}}, "count": {"$sum": 1}}},
    {"$match": {"count": {"$gt": 10}}},
    {"$sort": {"count": -1, "_id": 1}}
]

result_14 = list(db.methods.aggregate(pipeline))

pprint.pprint(result_14)
print(ordered_hash(result_14))


**Q15.** Compute the **number of methods per supervision type** (`Sup` field).  

In [19]:
# Uniformiser la casse et accepter les champs Sup ou sup.
pipeline = [
    {"$group": {
        "_id": {"$toLower": {"$ifNull": ["$Sup", "$sup"]}},
        "count": {"$sum": 1}
    }},
]

result_15= list(db.methods.aggregate(pipeline))

pprint.pprint(result_15)
print(result_hash(result_15))


[{'_id': 'semi-supervised', 'count': 59},
 {'_id': 'unsupervised', 'count': 73},
 {'_id': 'supervised', 'count': 1}]
bef0ca2dcc5abf1f


---------------------------------------------------------------------------------------

**Q16.** List all **distinct author names** across all method documents. How many unique authors are there in total?

In [20]:
def build_author_aliases(names):
    # Enlever les points et les espaces superflus ; ignorer la casse dans les cles.
    aliases = {
        " ".join(name.replace(".", "").split()).casefold(): " ".join(name.replace(".", "").split())
        for name in sorted(names)
    }

    # Indexer les noms complets par (initiale du prenom, nom de famille).
    full_names = {}
    for key, name in aliases.items():
        parts = key.split(" ", 1)
        prenom = parts[0]
        if len(parts) == 2 and len(prenom) > 1:
            signature = (prenom[0], parts[1])
            full_names.setdefault(signature, []).append(name)

    ambiguous_aliases = {}
    for key, name in list(aliases.items()):
        parts = key.split(" ", 1)
        prenom = parts[0]
        if len(parts) == 2 and len(prenom) == 1:
            signature = (prenom, parts[1])
            candidates = full_names.get(signature, [])
            if len(candidates) == 1:
                aliases[key] = candidates[0]
            elif len(candidates) > 1:
                ambiguous_aliases[name] = candidates

    return aliases, ambiguous_aliases


def standardize_author(name, aliases):
    cleaned = " ".join(name.replace(".", "").split())
    return aliases.get(cleaned.casefold(), cleaned)


In [21]:
pipeline = [
    {"$unwind": "$authors"},
    {"$group": {"_id": "$authors"}}
]

result_16 = [doc["_id"] for doc in db.methods.aggregate(pipeline)]
aliases, ambiguous_aliases = build_author_aliases(result_16)

print("Rapprochements automatiques :")
pprint.pprint({
    name: canonical for name, canonical in aliases.items()

})
if ambiguous_aliases:
    print("Noms ambigus, initiales conservees :")
    pprint.pprint(ambiguous_aliases)

result_16 = sorted(
    {standardize_author(name, aliases) for name in result_16},
    key=lambda name: (name.split()[-1].casefold(), name.casefold())
)
pprint.pprint(result_16)
print("Nombre d'auteurs uniques :", len(result_16))
print(result_hash(result_16))


Rapprochements automatiques :
{'a dengel': 'Andreas Dengel',
 'a smola': 'A Smola',
 'abdul fatir ansari': 'Abdul Fatir Ansari',
 'abdullah mueen': 'Abdullah Mueen',
 'abhimanyu das': 'Abhimanyu Das',
 'abhishek sharma': 'Abhishek Sharma',
 'ada fu': 'Ada Fu',
 'ada wai-chee fu': 'Ada Wai-Chee Fu',
 'aditya krishna menon': 'Aditya Krishna Menon',
 'aidan n gomez': 'Aidan N Gomez',
 'aidong men': 'Aidong Men',
 'akshat kumar': 'Akshat Kumar',
 'alberto ogbechie': 'Alberto Ogbechie',
 'aleksandar lazarevic': 'Aleksandar Lazarevic',
 'alexander lavin': 'Alexander Lavin',
 'ali caner turkmen': 'Ali Caner Turkmen',
 'alice chuang': 'Alice Chuang',
 'anastasios gounaris': 'Anastasios Gounaris',
 'andreas dengel': 'Andreas Dengel',
 'andreas terzis': 'Andreas Terzis',
 'andrei manolache': 'Andrei Manolache',
 'andrew gordon wilson': 'Andrew Gordon Wilson',
 'anguo zhang': 'Anguo Zhang',
 'anusha ramakrishnan': 'Anusha Ramakrishnan',
 'apostolos n papadopoulos': 'Apostolos N Papadopoulos',
 'a

Rq : in the data base there are name for which we don't really know who it is such as Z.Li that can be several person : Z. Li',
 'Zeyan Li',
 'Zheng Li',
 'Zhihan Li',
 'Ziyuan Li'

Also name like  ada Fu and Ada Wai-chee Fu might refer at the same person but i am not sure, so i won't include this case.
Also case where the second name might be written or not are not treated here but it doesn't look like it appears in the database

**Q17.** For each category, compute the **average publication year**. Sort by average year descending.

In [22]:
pipeline = [
    {"$group": {
        "_id": {"$toLower": {"$trim": {"input": "$category"}}},
        "average_year": {"$avg": "$year"}
    }},
    {"$sort": {"average_year": -1}}
]

result_17 = list(db.methods.aggregate(pipeline))

pprint.pprint(result_17)
print(ordered_hash(result_17))


[{'_id': 'reconstruction-based', 'average_year': 2020.4516129032259},
 {'_id': 'discord-based and matrix-profile-based', 'average_year': 2020.0},
 {'_id': 'discord-based', 'average_year': 2016.6666666666667},
 {'_id': 'tree-based', 'average_year': 2015.75},
 {'_id': 'graph-based', 'average_year': 2014.0},
 {'_id': 'encoding-based', 'average_year': 2013.0666666666666},
 {'_id': 'forecasting-based', 'average_year': 2012.4444444444443},
 {'_id': 'clustering-based', 'average_year': 2010.375},
 {'_id': 'distribution-based', 'average_year': 2009.5714285714287},
 {'_id': 'proximity-based', 'average_year': 2008.25},
 {'_id': 'distance-based', 'average_year': 2000.0}]
a7aac222ea06ccb5


**Q18.** Count the number of **streaming** methods per category. Only show categories with at least one streaming method. Sort in descending order based on the number of streaming methods per category. In case of equality, sort by lexicographical order based on the category name.

In [23]:
pipeline = [
    {"$match": {"Stream": True}},
    {"$group": {"_id": {"$toLower": {"$trim": {"input": "$category"}}}, "count": {"$sum": 1}}},
    {"$sort": {"count": -1, "_id": 1}}
]

result_18 = list(db.methods.aggregate(pipeline))

pprint.pprint(result_18)
print(ordered_hash(result_18))


[{'_id': 'forecasting-based', 'count': 19},
 {'_id': 'reconstruction-based', 'count': 11},
 {'_id': 'distribution-based', 'count': 4},
 {'_id': 'proximity-based', 'count': 4},
 {'_id': 'discord-based', 'count': 3},
 {'_id': 'clustering-based', 'count': 2},
 {'_id': 'discord-based and matrix-profile-based', 'count': 1},
 {'_id': 'encoding-based', 'count': 1}]
9b820a7893093fc0


**T3**. We want to build a **search engine** on top of the `methods` and
`families` collections. A user types a free-text query in natural language,
for instance:

- *unsupervised streaming methods based on distance by author X.*

and the engine must return the methods sorted from the most to the least relevant.

This is an **open question**. Explain how you would compute the
**relevance score** of each method for a given query, and therefore the ranking.

*Your answer here*

we could think of a method inspired by the course 1 where each document method would be represented as a vector we would build using 1 or 0 if a each word appears in the description of the method (all the fiefs and value). we could build the vocabulary using every word that are constrained (like Supervised, Unsupervised) every author name etc and + sample word that appears a lot in some document and not at all in others (we could compute the originality of a word by log() number of doc / number of doc it appears) multiplied by frequency of the word in the doc (like in t). And for eacf-idf. Then for every method we add (like in tf-idf), we could increase our vocabulary by adding the k most original word to the vocab. We can then compute the similarity between a vector query and every vector of the database. 

Another method would be to put a vocabulary for each field (for example vocab for Dim is "Supervised", "Unsupervised" and "Semi-SUpervised"). when a word appears in the query, we try to find it in each vocabulary (we could lower case etc ...) and if it exist in a vocab linked to a fiel, we query the database only for this word at this given field (for example find method with dim = Supervised). We do it for every word of the query and count for each document the number of time they appear in the result of the query. 

Another method today would be to use a llm to give the query a good form we could then use to query the database. for example a json saying { author : ... , d} and then query the database with this

## Section D — Updates and Schema Validation


**Q19.** Add a field `era` to every method document: **`"classic"`** for methods published before 2010, **`"modern"`** for 2010 or later. Verify by counting documents in each era.

In [24]:
db.methods.update_many(
    {"year": {"$lt": 2010}},
    {"$set": {"era": "classic"}}
)
db.methods.update_many(
    {"year": {"$gte": 2010}},
    {"$set": {"era": "modern"}}
)
pipeline = [
    {"$group": {"_id": "$era", "count": {"$sum": 1}}}
]
result_19 = list(db.methods.aggregate(pipeline))

pprint.pprint(result_19)
print(result_hash(result_19))


[{'_id': 'classic', 'count': 37}, {'_id': 'modern', 'count': 96}]
8f23f0e13d1f770a


**Q20.** Create a new collection `validated_methods` with a `$jsonSchema` validator enforcing:  
(a) `name` is required and must be a string;  
(b) `year` is required and must be an integer;  
(c) `Dim`, if present, must be `"Univariate"` or `"Multivariate"`.  
Insert one valid and one invalid document to confirm the schema works.

In [28]:
from pymongo.errors import WriteError

validator = {
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["name", "year"],
        "properties": {
            "name": {"bsonType": "string"},
            "year": {"bsonType": ["int", "long"]},
            "Dim": {"bsonType": "string", "enum": ["Univariate", "Multivariate"]}
        }
    }
}

# Recreer la collection de test pour pouvoir reexecuter cette cellule.
db.validated_methods.drop()
db.create_collection("validated_methods", validator=validator)

valid_document = {"name": "Valid method", "year": 2020, "Dim": "Multivariate"}
db.validated_methods.insert_one(valid_document)
print("Document valide accepte.")

invalid_document = {"name": "Invalid method", "year": "2020", "Dim": "Multivariate"}
try:
    db.validated_methods.insert_one(invalid_document)
except WriteError as error:
    if error.code != 121:
        raise
    print("Document invalide refuse : year est une chaine au lieu d'un entier.")
else:
    raise AssertionError("Le document invalide aurait du etre refuse.")

result_20 = list(db.validated_methods.find({}, {"_id": False}))
pprint.pprint(result_20)
print(result_hash(result_20))


Document valide accepte.
Document invalide refuse : year est une chaine au lieu d'un entier.
[{'Dim': 'Multivariate', 'name': 'Valid method', 'year': 2020}]
4c0f55f3f949d2d6


---
# Part 2 — Advanced Take-Home Assignment

## Part 2 Setup — Enter Your Seed

Before answering H1–H7, run the cell below with **your personal seed number**. The latter is computed from your last name (i.e., family name).  
This reloads the `methods` collection with two extra fields:

| Field | Type | Description |
|---|---|---|
| `rating` | int [1–5] | Simulated peer-review quality score |
| `citations` | int [50–5000] | Simulated citation count |

> ⚠️ **These values are unique to your seed.** Make sure you use your own seed number.


In [32]:
# ── SET YOUR SEED HERE ───────────────────────────────────────────────
Your_name = "borderies"  # ← replace None with your last name (i.e, family name)
YOUR_SEED = int(hashlib.sha256(Your_name.strip().lower().encode('utf-8')).hexdigest(), 16) % 1000
print(YOUR_SEED)
# ─────────────────────────────────────────────────────────────────────


def _doc_rng(seed, name):
    key = f'{seed}::{name}'.encode()
    return random.Random(int(hashlib.sha256(key).hexdigest(), 16))

def enrich(doc, seed):
    doc = dict(doc)
    rng = _doc_rng(seed, doc['name'])
    doc['rating']    = rng.randint(1, 5)
    doc['citations'] = rng.randint(50, 5000)
    return doc

enriched = [enrich(m, YOUR_SEED) for m in methods]
db.methods.drop()
db.methods.insert_many(enriched)

# Quick sanity check
sample = db.methods.find_one({}, {'name':1,'rating':1,'citations':1,'_id':0})
print(f'Loaded {db.methods.count_documents({})} methods for seed {YOUR_SEED}')
print('Sample document:', sample)


578
Loaded 133 methods for seed 578
Sample document: {'name': 'AD-LTI', 'rating': 1, 'citations': 4932}


## Part 2 Questions


### H1 — Category Performance Summary

For each category, compute **(a)** the **average rating** (rounded to 2 decimal places) and **(b)** the **total number of citations**.  
Return only categories with **at least 5 methods**, sorted by average rating descending, then total citations descending.


In [33]:
pipeline = [
    {"$group": {
        "_id": {"$toLower": {"$trim": {"input": "$category"}}},
        "count": {"$sum": 1},
        "average_rating": {"$avg": "$rating"},
        "total_citations": {"$sum": "$citations"}
    }},
    {"$match": {"count": {"$gte": 5}}},
    {"$project": {
        "average_rating": {"$round": ["$average_rating", 2]},
        "total_citations": True
    }},
    {"$sort": {"average_rating": -1, "total_citations": -1}}
]

result_h1 = list(db.methods.aggregate(pipeline))

pprint.pprint(result_h1)
print(ordered_hash(result_h1))


[{'_id': 'proximity-based', 'average_rating': 3.5, 'total_citations': 23427},
 {'_id': 'forecasting-based', 'average_rating': 3.26, 'total_citations': 47971},
 {'_id': 'distribution-based',
  'average_rating': 3.24,
  'total_citations': 46582},
 {'_id': 'graph-based', 'average_rating': 2.8, 'total_citations': 16036},
 {'_id': 'clustering-based', 'average_rating': 2.75, 'total_citations': 18200},
 {'_id': 'encoding-based', 'average_rating': 2.67, 'total_citations': 42073},
 {'_id': 'discord-based', 'average_rating': 2.67, 'total_citations': 32861},
 {'_id': 'reconstruction-based',
  'average_rating': 2.52,
  'total_citations': 64376}]
19f9f0b47a7de23a


> ✏️ **Explain your approach** — after running your query above, briefly describe the steps of your pipeline in plain language.
> What stages did you use and why? What traps or edge cases did you watch out for?


We group by category (which we put to lower case without spaces with lower case and trim) and compute the average rating and total citation which are asked and the count to be able to make a match after. We then select according to the count to keep only >5 . Finally we only keep the 2 numbers asked and sort according to it 


### H2 — Prolific Cross-Category Authors

Find all authors who have co-authored papers in **at least 3 different categories**.  
For each such author, return their name and the **sorted list of distinct categories** they published in.  
Sort by number of categories descending, then by author name alphabetically.


In [34]:
author_names = db.methods.distinct("authors")
aliases_h2, ambiguous_aliases_h2 = build_author_aliases(author_names)

# Traduire les correspondances Python en conditions utilisables par MongoDB.
author_cases = [
    {
        "case": {"$eq": ["$authors", name]},
        "then": standardize_author(name, aliases_h2)
    }
    for name in author_names
]

pipeline = [
    {"$unwind": "$authors"},
    {"$project": {
        "author": {"$switch": {"branches": author_cases, "default": "$authors"}},
        "category": {"$toLower": {"$trim": {"input": "$category"}}}
    }},
    {"$group": {"_id": "$author", "categories": {"$addToSet": "$category"}}},
    {"$set": {
        "count": {"$size": "$categories"},
        "categories": {"$sortArray": {"input": "$categories", "sortBy": 1}}
    }},
    {"$match": {"count": {"$gte": 3}}},
    {"$sort": {"count": -1, "_id": 1}}
]

result_h2 = list(db.methods.aggregate(pipeline))

pprint.pprint(result_h2)
print(ordered_hash(result_h2))


[{'_id': 'Themis Palpanas',
  'categories': ['clustering-based',
                 'discord-based and matrix-profile-based',
                 'distribution-based',
                 'graph-based'],
  'count': 4},
 {'_id': 'Eamonn Keogh',
  'categories': ['discord-based',
                 'discord-based and matrix-profile-based',
                 'encoding-based'],
  'count': 3}]
71bb371d90c5b35c


> ✏️ **Explain your approach** — after running your query above, briefly describe the steps of your pipeline in plain language.
> What stages did you use and why? What traps or edge cases did you watch out for?


To deal with the different version of a name I have, i reuse the aliases i built at Q16 and I use it build many conditional step to replace the every name by the standardized name. We then use this list of conditional value to get the fiel author completed by our canonical author's name. (+we still assure that category is without space and lower case). This allow us to group by author name and also for each doc, in the process of grouping we keep a set of the different category it apears in in order to count it and match on it. we then need too keep track of the number of category and to order the category. Finally we only select the author that have >3 categorie and sort them according to this number and their id 

### H3 — Highest-Rated Category

Find the category where the **proportion of methods with rating ≥ 4** to the total number of methods is the highest.  
Tie-break: highest **total citations**. Return the category name, the proportion (rounded to 3 decimal places), and the total citations.


In [ ]:

# Your answer here. Print ordered_hash of the result.


> ✏️ **Explain your approach** — after running your query above, briefly describe the steps of your pipeline in plain language.
> What stages did you use and why? What traps or edge cases did you watch out for?


*Your explanation here…*


### H4 — Top Cited Methods in Recent High-Performing Categories

Consider only methods published **between 2015 and 2020 inclusive**.  
Among these, find the **top-3 categories** by average citations.  
For each of these 3 categories, return the **single method with the highest individual citation count** (name, category, citations, year).  



In [ ]:

# Your answer here. Print ordered_hash of the result.


> ✏️ **Explain your approach** — after running your query above, briefly describe the steps of your pipeline in plain language.
> What stages did you use and why? What traps or edge cases did you watch out for?


*Your explanation here…*


### H5 — Family–Method Cross-Collection Join

For each document in the **`families`** collection whose `name` matches a `category` in **`methods`**, return:  
(a) the family name, (b) the number of methods in that category,  
(c) the average rating (rounded to 2 decimal places),  
(d) whether any method in that category has a `snippet` field.  
Sort by average rating descending.  



In [ ]:

# Your answer here. Print ordered_hash of the result.


> ✏️ **Explain your approach** — after running your query above, briefly describe the steps of your pipeline in plain language.
> What stages did you use and why? What traps or edge cases did you watch out for?


*Your explanation here…*


### H6 — Most Productive Year

Find the **single year** in which the largest number of **distinct authors** published at least one method.  
Return that year, the count of distinct authors, and the **list of author names** (in lexicographical order).


In [ ]:

# Your answer here. Print ordered_hash of the result.


> ✏️ **Explain your approach** — after running your query above, briefly describe the steps of your pipeline in plain language.
> What stages did you use and why? What traps or edge cases did you watch out for?


*Your explanation here…*


### H7 — Supervision-Type Deep Dive

For each **supervision type** (`Sup` field), produce a summary with:  
(a) the supervision type, (b) total methods, (c) the **most common category**, (d) the average publication year (rounded to 1 decimal place),  
(e) the **names of the top-2 most cited methods**.  

Sort the results by total methods, descending.


In [ ]:

# Your answer here. Print ordered_hash of the result.


> ✏️ **Explain your approach** — after running your query above, briefly describe the steps of your pipeline in plain language.
> What stages did you use and why? What traps or edge cases did you watch out for?


*Your explanation here…*


### H8 — Co-authorship Shortest Path

Two authors are **co-authors** if they appear together in at least one method's `authors` list. This defines a graph where authors are nodes and a co-authorship link is an edge.

**(a)** Build a `co_authors` collection in MongoDB (one document per author, with a `co_authors` list of their collaborators).  
Then write a `$graphLookup` aggregation that tries to find the **shortest path** between two specific authors — including all intermediate nodes.  
 

**(b)** Implement a Python function `find_path(start, end)` that returns the **shortest path** as a list of author names.  
Test it on the following pairs:

| Pair | Author A | Author B |
|---|---|---|
| 1 | `'Eamonn Keogh'` | `'John Paparrizos'` |
| 2 | `'Eamonn Keogh'` | `'D. Papadopoulos'` |
| 3 | `'Pavel Senin'` | `'D. Papadopoulos'` |

**(c)** Discuss the feasibility, the advantages and the limits of both (a) and (b). Can you imagine an alternative to (a) and (b)?


#### (a) Build `co_authors` and attempt `$graphLookup`


In [ ]:
# Step 1: build co-authorship adjacency dict from the methods collection
from collections import deque

adjacency = {}
# Your code here

# Step 2: insert into a co_authors collection
db.co_authors.drop()
# Your code here

# Step 3: try $graphLookup to retrieve the path
# (choose any two authors for the demo)
start_author = 'Eamonn Keogh'
end_author   = '???'   # ← pick an end author

gl_result = list(db.co_authors.aggregate([
    # Your $graphLookup pipeline here
]))
print(gl_result)


#### (b) Python — shortest co-authorship path


In [ ]:
def find_path(start, end):
    """Return the shortest co-authorship path as a list of author names, or None."""
    # Your implementation here
    pass

# Test on the required pairs
all_sorted = sorted(adjacency.keys())

test_pairs = [
    ('Eamonn Keogh', 'John Paparrizos'),   
    ('Eamonn Keogh', 'D. Papadopoulos'),  
    ('Pavel Senin',  'D. Papadopoulos'),  
]

for a, b in test_pairs:
    path, err = find_path(a, b)
    if path:
        print(f'[{len(path)-1} hop(s)]  {" → ".join(path)}')
    else:
        print(f'  {a!r} ↔ {b!r}: {err}')


> ✏️ **Explain your approach** — after running your query above, briefly describe the steps of your pipeline in plain language.
> What stages did you use and why? What traps or edge cases did you watch out for?


*Your explanation here…*


#### (c) Discuss the feasibility, the advantages and the limits of both (a) and (b). Can you imagine an alternative to (a) and (b)?


*Your explanation here…*


### H9 — Academic Descendants (Design Question)

Consider the following query:

> *"Which author has the most descendants that published at least one paper in the database?"*

Here, **descendant** means an academic descendant in the sense of the [Math Genealogy Project](https://genealogy.math.ndsu.nodak.edu/): a PhD student advised by the author, or a student of a student, etc.

**1. Can you answer this query directly with the current database?** Explain your answer.

**2. Propose a high-level solution.** Describe, in plain language or pseudo-code, your solution (building the advisor–student collection and querying it).


*Write your answer here (plain text / pseudo-code)…*
